# Web Scraper Coral Taxonomy

A webscrapper to get data on coral description (text), image, color (text)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from typing import Optional, List, Dict
import time
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains

import os

In [2]:
# Set headers to mimic a real browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def fetch_page(url: str, timeout: int = 10) -> Optional[requests.Response]:
    """
    Fetch a webpage and return the response object.
    
    Args:
        url: URL to scrape
        timeout: Request timeout in seconds
    
    Returns:
        requests.Response object if successful, None otherwise
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()  # Raise exception for bad status codes
        return response
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

## Fetch the Data


## Handling Dynamic Content & Expandable Trees

For websites with JavaScript-driven trees (like jstree), use Selenium or Playwright to automate clicking

In [3]:
def scrape_coral_species(url: str) -> List[Dict]:
    """
    Scrape coral species from expandable tree structure.
    Expands all collapsed nodes one at a time, then collects all species links.
    
    Args:
        url: URL to scrape (https://www.coralsoftheworld.org/species_factsheets/)
    
    Returns:
        List of dictionaries with species data (name, url)
    """
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    # Uncomment for headless mode (faster, no browser window):
    # chrome_options.add_argument("--headless")
    
    driver = webdriver.Chrome(options=chrome_options)
    results = []
    
    try:
        driver.get(url)
        time.sleep(3)  # Wait for initial page load and jstree initialization
        
        wait = WebDriverWait(driver, 10)
        
        # Wait for tree container to be present
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.jstree-node")))
        
        expanded_count = 0
        
        # Keep expanding until no more collapsed nodes exist
        max_iterations = 300
        iteration = 0
        
        print("Starting to expand all tree nodes...")
        
        while iteration < max_iterations:
            # Find collapsed nodes - refresh this list each time
            collapsed_nodes = driver.find_elements(By.CSS_SELECTOR, "li.jstree-closed")
            
            if not collapsed_nodes:
                print(f"✓ All {expanded_count} nodes expanded successfully!")
                break
            
            iteration += 1
            first_closed = collapsed_nodes[0]
            
            # Try multiple strategies to expand
            success = False
            
            # Strategy 1: Click the icon with JavaScript
            try:
                icon = first_closed.find_element(By.CSS_SELECTOR, "i.jstree-icon")
                driver.execute_script("arguments[0].scrollIntoView(true);", first_closed)
                time.sleep(0.1)
                driver.execute_script("arguments[0].click();", icon)
                success = True
            except Exception as e1:
                # Strategy 2: Click the anchor directly with JavaScript
                try:
                    anchor = first_closed.find_element(By.CSS_SELECTOR, "a.jstree-anchor")
                    driver.execute_script("arguments[0].click();", anchor)
                    success = True
                except Exception as e2:
                    # Strategy 3: Double-click the anchor
                    try:
                        anchor = first_closed.find_element(By.CSS_SELECTOR, "a.jstree-anchor")
                        ActionChains(driver).double_click(anchor).perform()
                        success = True
                    except Exception as e3:
                        pass
            
            if success:
                expanded_count += 1
                if expanded_count % 20 == 0:
                    print(f"  ... expanded {expanded_count} nodes so far")
                time.sleep(0.15)  # Wait for expansion animation
            else:
                # Skip problematic node and try next
                time.sleep(0.2)
        
        print(f"\nTotal nodes expanded: {expanded_count}")
        
        # Now scrape all visible links from the fully expanded tree
        print("Scraping all visible species links...")
        time.sleep(1)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Find all anchor tags in the tree
        #all_anchors = soup.find_all('a', class_='jstree-anchor')
        all_anchors = soup.find_all('li', class_='jstree-leaf')
        all_anchors = [li.find('a', class_='jstree-anchor') for li in all_anchors]
        print(f"Found {len(all_anchors)} total anchors in tree")
        
        for anchor in all_anchors:
            species_name = anchor.get_text(strip=True)
            
            if species_name:
                species_url = "https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/" + species_name.lower().replace(" ", "-") + "/"
                print(species_url)
                results.append({
                    'name': species_name,
                    'url': species_url
                })
        
        print(f"\n✓ Total species collected: {len(results)}")
        
        return results
    
    finally:
        driver.quit()

In [5]:
# Scrape all coral species from the expandable tree
url = "https://www.coralsoftheworld.org/species_factsheets/"
coral_species = scrape_coral_species(url)

# Convert to DataFrame for analysis
df_species = pd.DataFrame(coral_species)
print(f"\nTotal species pages to visit: {len(df_species)}")
print("\nFirst 10 species:")
print(df_species.head(10))

Starting to expand all tree nodes...
  ... expanded 20 nodes so far
  ... expanded 40 nodes so far
  ... expanded 60 nodes so far
  ... expanded 80 nodes so far
  ... expanded 100 nodes so far
  ... expanded 120 nodes so far
✓ All 122 nodes expanded successfully!

Total nodes expanded: 122
Scraping all visible species links...
Found 831 total anchors in tree
https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-bowerbanki/
https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-brevis/
https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-echinata/
https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-faviaformis/
https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-hemprichii/
https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-hillae/
https://www.coralsoftheworld.o

In [6]:
def scrape_species_details(species_url: str) -> Optional[Dict]:
    """
    Scrape detailed information from an individual species page.
    
    Args:
        species_url: URL of the species factsheet
    
    Returns:
        Dictionary with species details or None if failed
    """
    try:
        driver = webdriver.Chrome()
        driver.get(species_url)
        driver.implicitly_wait(5)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        driver.quit()
        
        # Extract title
        name = soup.find('h3').get_text(strip=True)
        
        # Extract description/content
        characters_p = soup.find('p', class_='mini-vertical-buffer')

        # Extract image URL from dynamically loaded content
        image_url = None
        image_div = soup.find('div', class_='image_preview')
        if image_div:
            image_a = image_div.find('img')
            if image_a:
                image_url = "https://www.coralsoftheworld.org" + image_a['src']

        color = None
        habitat = None
        abundance = None
        if characters_p:
            characters = characters_p.get_text(strip=True)
            characters = characters.replace("Characters: ", "", 1)

            color_p = characters_p.find_next_sibling('p')
            if color_p:
                color = color_p.get_text(strip=True)
                color = color.replace("Colour: ", "", 1)
                habitat_p = color_p.find_next_sibling('p').find_next_sibling('p')
                if habitat_p:
                    habitat = habitat_p.get_text(strip=True)
                    habitat = habitat.replace("Habitat: ","",1)
                    abundance_p = habitat_p.find_next_sibling('p')
                    if abundance_p:
                        abundance = abundance_p.get_text(strip=True)
                        abundance = abundance.replace("Abundance: ","",1)
        else:
            characters = None
        
        return {
            'species': name,
            'url': species_url,
            'characters': characters,
            'color': color,
            'habitat': habitat,
            'abundance': abundance,
            'image_url': image_url
        }
    except Exception as e:
        print(f"Error scraping {species_url}: {e}")
        return None


def visit_all_species(species_list: List[Dict]) -> List[Dict]:
    """
    Visit each species subpage and scrape details.
    
    Args:
        species_list: List of species dictionaries from scrape_coral_species()
    
    Returns:
        List of dictionaries with full species details
    """
    detailed_species = []
    
    print(f"Visiting {len(species_list)} species pages...")
    
    for idx, specie in enumerate(species_list):
        print(f"\n[{idx + 1}/{len(species_list)}] Scraping: {specie['name']}")
        print(f"  URL: {specie['url']}")
        
        details = scrape_species_details(specie['url'])
        if details:
            detailed_species.append(details)
            time.sleep(1)  # Be respectful to the server
        else:
            print(f"  Failed to scrape")
    
    return detailed_species

In [7]:
# Visit each species page to extract detailed information
# WARNING: This will take a while - set a limit for testing

# Visit only first 5 species for testing:
species_sample = df_species.head(3).to_dict('records')
detailed_data = visit_all_species(df_species.to_dict('records'))

df_detailed = pd.DataFrame(detailed_data)
print("\n" + "="*60)
print("Species Details")
print("="*60)
print(df_detailed)



Visiting 831 species pages...

[1/831] Scraping: Acanthastrea bowerbanki
  URL: https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-bowerbanki/

[2/831] Scraping: Acanthastrea brevis
  URL: https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-brevis/

[3/831] Scraping: Acanthastrea echinata
  URL: https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-echinata/

[4/831] Scraping: Acanthastrea faviaformis
  URL: https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-faviaformis/

[5/831] Scraping: Acanthastrea hemprichii
  URL: https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-hemprichii/

[6/831] Scraping: Acanthastrea hillae
  URL: https://www.coralsoftheworld.org/species_factsheets/species_factsheet_summary/acanthastrea-hillae/

[7/831] Scraping: Acanthastrea ishigakiensis
  URL: https://www.coral

In [8]:
# Save to CSV

output_dir = 'scrapped_data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'coral_species.csv')
df_detailed.to_csv(output_path, index=False)
print(f"\n✓ Data saved to: {output_path}")


✓ Data saved to: scrapped_data\coral_species.csv
